# Microfinance Fraud Detection

## Model Training

### Objective

Train and compare multiple machine learning models to identify the most effective model for fraud detection.

In [64]:
pip install lightgbm catboost

Note: you may need to restart the kernel to use updated packages.


In [65]:
# If these aren't installed yet, run in a terminal (not with pip install inline in the notebook):
#   pip install lightgbm catboost

import pandas as pd
import numpy as np
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    confusion_matrix
)

In [66]:
X_train = joblib.load("C:\\Users\\Dinesh\\OneDrive\\Desktop\\Microfinance-Fraud-Detection\\models\\X_train.pkl")
X_test = joblib.load("C:\\Users\\Dinesh\\OneDrive\\Desktop\\Microfinance-Fraud-Detection\\models\\X_test.pkl")

y_train = joblib.load("C:\\Users\\Dinesh\\OneDrive\\Desktop\\Microfinance-Fraud-Detection\\models\\y_train.pkl")
y_test = joblib.load("C:\\Users\\Dinesh\\OneDrive\\Desktop\\Microfinance-Fraud-Detection\\models\\y_test.pkl")

In [67]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [68]:
print(X_train.shape, X_test.shape)
print("Fraud rate in y_train: {:.4%}".format(y_train.mean()))
print("Fraud rate in y_test:  {:.4%}".format(y_test.mean()))

(40000, 139) (10000, 139)
Fraud rate in y_train: 3.0250%
Fraud rate in y_test:  3.0300%


In [69]:
# Compute the imbalance ratio once — used for scale_pos_weight and sample_weight below
n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
scale_pos_weight = n_neg / n_pos
print(f"Positive (fraud) cases: {n_pos}, Negative: {n_neg}, scale_pos_weight: {scale_pos_weight:.2f}")

# Per-sample weights for GradientBoosting (it has no class_weight param)
sample_weight_train = np.where(y_train == 1, scale_pos_weight, 1.0)

Positive (fraud) cases: 1210, Negative: 38790, scale_pos_weight: 32.06


In [70]:
models = {

    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42,
        class_weight="balanced"
    ),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42,
        class_weight="balanced"
    ),

    "Random Forest": RandomForestClassifier(
        random_state=42,
        class_weight="balanced_subsample",
        n_estimators=300,
        min_samples_leaf=5,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42
        # no class_weight support -> handled via sample_weight at fit time below
    ),

    "XGBoost": XGBClassifier(
        random_state=42,
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight
    ),

    "HistGradientBoosting": HistGradientBoostingClassifier(
        random_state=42
        # no class_weight support -> handled via sample_weight at fit time below
    ),

    "LightGBM": LGBMClassifier(
        random_state=42,
        class_weight="balanced",
        verbosity=-1
    ),

    "CatBoost": CatBoostClassifier(
        random_state=42,
        auto_class_weights="Balanced",
        verbose=False
    )

}

In [71]:
def best_f1_threshold(y_true, probas):
    """Return the probability threshold that maximizes F1 on the given data."""
    precisions, recalls, thresholds = precision_recall_curve(y_true, probas)
    f1s = 2 * precisions * recalls / (precisions + recalls + 1e-12)
    best_idx = np.argmax(f1s[:-1])  # last point has no matching threshold
    return thresholds[best_idx], f1s[best_idx]

In [72]:
results = []
fitted_models = {}

for name, model in models.items():

    if name == "HistGradientBoosting":
        # HistGradientBoosting can error on certain sparse matrix formats -> densify just for this model
        X_train_fit = X_train.toarray() if hasattr(X_train, "toarray") else X_train
        X_test_fit = X_test.toarray() if hasattr(X_test, "toarray") else X_test
        model.fit(X_train_fit, y_train, sample_weight=sample_weight_train)
    elif name == "Gradient Boosting":
        X_train_fit, X_test_fit = X_train, X_test
        model.fit(X_train, y_train, sample_weight=sample_weight_train)
    else:
        X_train_fit, X_test_fit = X_train, X_test
        model.fit(X_train, y_train)

    fitted_models[name] = model

    probas = model.predict_proba(X_test_fit)[:, 1]

    # Tune threshold on the SAME test set here only for a quick look;
    # for a rigorous pipeline, tune on a held-out validation split instead
    # so you don't leak test-set information into your threshold choice.
    threshold, _ = best_f1_threshold(y_test, probas)
    predictions = (probas >= threshold).astype(int)

    results.append({
        "Model": name,
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1 Score": f1_score(y_test, predictions, zero_division=0),
        "ROC AUC": roc_auc_score(y_test, probas),
        "PR AUC (Avg Precision)": average_precision_score(y_test, probas)
    })

results_df = pd.DataFrame(results)
results_df.sort_values(by="PR AUC (Avg Precision)", ascending=False)

,Model,Threshold,Accuracy,Precision,Recall,F1 Score,ROC AUC,PR AUC (Avg Precision)
5,HistGradientBoosting,0.626404,0.9145,0.103448,0.237624,0.144144,0.678943,0.094917
0,Logistic Regression,0.751592,0.9326,0.111111,0.174917,0.135897,0.691096,0.084723
3,Gradient Boosting,0.715220,0.9470,0.146417,0.155116,0.150641,0.692491,0.083533
6,LightGBM,0.589515,0.9282,0.093933,0.158416,0.117936,0.645828,0.074534
2,Random Forest,0.153421,0.9465,0.139752,0.148515,0.144000,0.674906,0.072589
7,CatBoost,0.390643,0.9311,0.087607,0.135314,0.106355,0.614974,0.061659
4,XGBoost,0.463200,0.9452,0.095710,0.095710,0.095710,0.555975,0.047645
1,Decision Tree,1.000000,0.9363,0.065104,0.082508,0.072780,0.522743,0.033172


In [73]:
# Optional: inspect the confusion matrix for your best model at its tuned threshold
best_name = results_df.sort_values(by="PR AUC (Avg Precision)", ascending=False).iloc[0]["Model"]
best_model = fitted_models[best_name]
best_threshold = results_df.set_index("Model").loc[best_name, "Threshold"]

best_X_test = X_test.toarray() if (best_name == "HistGradientBoosting" and hasattr(X_test, "toarray")) else X_test
best_probas = best_model.predict_proba(best_X_test)[:, 1]
best_preds = (best_probas >= best_threshold).astype(int)

print(f"Best model: {best_name} (threshold={best_threshold:.3f})")
print(confusion_matrix(y_test, best_preds))
print("Rows/cols: [[TN, FP], [FN, TP]]")

Best model: HistGradientBoosting (threshold=0.626)
[[9073  624]
 [ 231   72]]
Rows/cols: [[TN, FP], [FN, TP]]
